# Практическая работа №2: Обработка выборочных данных. Нахождение точечных оценок параметров распределения

Выполнили студенты гр. 2384 Тимченко Дмитрий и Цыганков Роман.

## Цель работы
Получение практических навыков нахождения точечных статистических оценок параметров распределения.

## Основные теоретические положения
**Точечная оценка** - это приближенное значение неизвестного параметра генеральной совокупности, вычисленное по выборочным данным. Надежная точечная оценка должна удовлетворять свойствам:
- **Несмещенность** - математическое ожидание оценки равно оцениваемому параметру;
- **Состоятельность** - оценка сходится по вероятность к параметру при увеличении объема выборки;
- **Эффективность** - оценка имеет наименьшую дисперсию среди всех оценок.

**Начальный эмпирический момент k-го порядка:**
$\bar{M_k} = \frac{1}{N}\sum n_jx_j^k$, где

$N$ - объем выборки,

$n_j$ - абсолютная частота,

$x_j$ - середина интервала.


**Центральный эмпирический момент k-ого порядка:**
$\bar{m_k} = \frac{1}{N}\sum n_j(x_j - \bar{x_в})^k$,

где $\bar{x_в} = \bar{M_1} = \frac{1}{N}\sum n_jx_j$.

**Выборочная дисперсия**: $D_в = \bar{m_2} = \frac{1}{N}\sum n_j(x_j - \bar{x_в})^2$


**Исправленная оценка дисперсии:** $s^2 = \frac{N}{N-1}D_в$

**Статистическая оценка СКО:** $\sigma_в = \sqrt D_в$

**Статистическая оценка асимметрии:** $\bar{A_s} = \frac{\bar{m_3}}{s^3}$

**Статистическая оценка эксцесса:** $\bar{E} = \frac{\bar{m_4}}{s^4} - 3$

**Метод условных вариант** используется для упрощения вычислений при обработке интервальных рядов. Суть метода заключается в переходе к новым переменным - условным вариантам:
$u_j=\frac{x_j - C}{h}$ , где:

$x_j$ — середина j-го интервала

$C$ — условный нуль

$h$ — шаг

**Условный момент к-ого порядка:** $\bar{M_k}^* = \frac{1}{N}\sum n_j (\frac{x_j - C}{h})^k = \frac{1}{N}\sum n_j u_j^k$

### Робастные выборочные статистические оценки:
1. Модой вариационного ряда называется варианта, которой соответсвует наибольшая частота. $M_o^* = x_{i-1} + h \frac{\tilde{m_i} - \tilde{m_{i-1}}}{(\tilde{m_i} - \tilde{m_{i-1}}) + (\tilde{m_i} - \tilde{m_{i+1}})}$, где $x_{i-1}$ - левая граница интервала, $h$ - длина интервала в интервальном ряде, $\tilde{m_i}$ - относительная частота для i-ого интервала.
2. Медианой называется варианта, соответсвующая середине ранжированного ряда. $M_e^* = x_{i-1} + \frac{h}{\tilde{m_i}}(0.5 - \sum_{j=1}^{i-1} \tilde{m_j})$
3. Выборочный коэффициент вариации характеризует разброс значений вариант относительно выборочного среднего: $V^* = \frac{\sigma_в}{\bar{x_в}} \cdot 100 \%$

## Постановка задачи
Для заданных выборочных данных вычислить с использованием метода моментов и условных вариант точечные статистические оценки математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации исследуемой случайной величины. Полученные результаты содержательно проинтерпретировать.

## Выполнение работы

### 1. Для середин интервального ряда, полученного в практической работе №1, вычислить условные варианты. Заполнить таблицу (в последней строке $\sum$ необходимо заполнить суммы столбцов; ячейки отмеченные прочерком заполнять не надо). Провести контроль вычислений.

Наработки из первой практической работы:

In [28]:
import pandas as pd
import numpy as np

# 1. Загрузка данных
# skiprows=3, так как в начале файла 3 строки комментариев
df = pd.read_csv('sample.csv', skiprows=3)

# 2. Формирование двумерной выборки (первые 108 строк)
# Выбираем столбцы 'nu' (объемный вес) и 'E' (модуль упругости)
sample_2d = df[['nu', 'E']].head(108)

# 3. Проверка объема
print(f"Размер двумерной выборки: {sample_2d.shape}")
print("\nПервые 5 объектов выборки (пара признаков):")
print(sample_2d.head())

Размер двумерной выборки: (108, 2)

Первые 5 объектов выборки (пара признаков):
    nu      E
0  480  153.3
1  510  129.4
2  426  119.0
3  482  139.9
4  393  103.2


In [30]:
# Извлечем данные в массивы numpy для удобства
data_nu = sample_2d['nu'].values
data_e = sample_2d['E'].values

In [42]:
from IPython.display import display, Markdown
def create_interval_table(data, n, feature_name):
    """Создает интервальную таблицу, аналогично ЛР №1."""
    k = round(1 + 3.322 * np.log10(n))
    x_min, x_max = data.min(), data.max()
    h = (x_max - x_min) / k
    bins = np.linspace(x_min, x_max, k + 1)

    intervals = []
    for i in range(k):
        start, end = bins[i], bins[i+1]
        if i == k - 1:
            m_i = np.sum((data >= start) & (data <= end))
        else:
            m_i = np.sum((data >= start) & (data < end))
        mid_x = (start + end) / 2
        intervals.append({
            'i': i + 1,
            '$[x_i, x_{i+1})$': f"[{start:.2f}, {end:.2f})",
            '$\\tilde{x}_i$': round(mid_x, 2),
            '$m_i$': int(m_i),
        })

    df_intervals = pd.DataFrame(intervals)
    # Добавим строку итогов для полноты
    total_row = pd.DataFrame([{
        'i': 'Σ', '$[x_i, x_{i+1})$': '-', '$\\tilde{x}_i$': '-',
        '$m_i$': df_intervals['$m_i$'].sum()
    }])
    final_table = pd.concat([df_intervals, total_row], ignore_index=True)
    return final_table.iloc[:-1].copy(), h

# Построим таблицы для nu и E
nu_interval_data, h_nu = create_interval_table(data_nu, 108, "nu")
e_interval_data, h_e = create_interval_table(data_e, 108, "E")

print("Исходные данные для nu:")
display(Markdown(nu_interval_data[['i', '$\\tilde{x}_i$', '$m_i$']].to_markdown(index=False)))
print(f"Шаг интервала h_nu = {h_nu:.4f}")

print("\nИсходные данные для E:")
display(Markdown(e_interval_data[['i', '$\\tilde{x}_i$', '$m_i$']].to_markdown(index=False)))
print(f"Шаг интервала h_e = {h_e:.4f}")

Исходные данные для nu:


|   i |   $\tilde{x}_i$ |   $m_i$ |
|----:|----------------:|--------:|
|   1 |          337.06 |       9 |
|   2 |          371.19 |       4 |
|   3 |          405.31 |      24 |
|   4 |          439.44 |      21 |
|   5 |          473.56 |      23 |
|   6 |          507.69 |      16 |
|   7 |          541.81 |       7 |
|   8 |          575.94 |       4 |

Шаг интервала h_nu = 34.1250

Исходные данные для E:


|   i |   $\tilde{x}_i$ |   $m_i$ |
|----:|----------------:|--------:|
|   1 |           72.18 |       5 |
|   2 |           87.54 |       7 |
|   3 |          102.91 |      14 |
|   4 |          118.27 |      23 |
|   5 |          133.63 |      27 |
|   6 |          148.99 |      22 |
|   7 |          164.36 |       7 |
|   8 |          179.72 |       3 |

Шаг интервала h_e = 15.3625


In [50]:
def calculate_conditional_variants(data_table, h, feature_name):
    """Вычисляет условные варианты u_i и необходимые суммы."""
    df = data_table.copy()
    x_mid = df['$\\tilde{x}_i$'].values
    m_i = df['$m_i$'].values

    # Выбираем условный нуль C (середина интервала с максимальной частотой)
    max_freq_idx = df['$m_i$'].idxmax()
    C = df.loc[max_freq_idx, '$\\tilde{x}_i$']

    print(f"ВЫЧИСЛЕНИЕ УСЛОВНЫХ ВАРИАНТ ДЛЯ ПРИЗНАКА: {feature_name}")
    print(f"Шаг интервала h = {h:.4f}")
    print(f"Условный нуль C = {C:.2f} (середина интервала с максимальной частотой)")

    # условные варианты u_i
    df['$u_i$'] = ((df['$\\tilde{x}_i$'] - C) / h).round(3)

    df['$n_i u_i$'] = (df['$m_i$'] * df['$u_i$']).round(3)
    df['$n_i u_i^2$'] = (df['$m_i$'] * df['$u_i$']**2).round(3)
    df['$n_i u_i^3$'] = (df['$m_i$'] * df['$u_i$']**3).round(3)
    df['$n_i u_i^4$'] = (df['$m_i$'] * df['$u_i$']**4).round(3)
    df['$n_i (u_i+1)^4$'] = (df['$m_i$'] * (df['$u_i$'] + 1)**4).round(3)

    result_table = df[['i', '$\\tilde{x}_i$', '$m_i$', '$u_i$',
                       '$n_i u_i$', '$n_i u_i^2$', '$n_i u_i^3$',
                       '$n_i u_i^4$', '$n_i (u_i+1)^4$']].copy()

    sums = pd.DataFrame({
        'i': ['Σ'],
        '$\\tilde{x}_i$': ['-'],
        '$m_i$': [df['$m_i$'].sum()],
        '$u_i$': ['-'],
        '$n_i u_i$': [result_table['$n_i u_i$'].sum()],
        '$n_i u_i^2$': [result_table['$n_i u_i^2$'].sum()],
        '$n_i u_i^3$': [result_table['$n_i u_i^3$'].sum()],
        '$n_i u_i^4$': [result_table['$n_i u_i^4$'].sum()],
        '$n_i (u_i+1)^4$': [result_table['$n_i (u_i+1)^4$'].sum()]
    })

    final_table = pd.concat([result_table, sums], ignore_index=True)
    return final_table, C, df['$m_i$'].sum()

# Вычисляем для nu
nu_conditional_table, nu_C, nu_n = calculate_conditional_variants(nu_interval_data, h_nu, "nu")
print("\nТаблица условных вариант для nu:")
display(Markdown(nu_conditional_table.to_markdown(index=False)))

# Вычисляем для E
e_conditional_table, e_C, e_n = calculate_conditional_variants(e_interval_data, h_e, "E")
print("\nТаблица условных вариант для E:")
display(Markdown(e_conditional_table.to_markdown(index=False)))

ВЫЧИСЛЕНИЕ УСЛОВНЫХ ВАРИАНТ ДЛЯ ПРИЗНАКА: nu
Шаг интервала h = 34.1250
Условный нуль C = 405.31 (середина интервала с максимальной частотой)

Таблица условных вариант для nu:


| i   | $\tilde{x}_i$   |   $m_i$ | $u_i$               |   $n_i u_i$ |   $n_i u_i^2$ |   $n_i u_i^3$ |   $n_i u_i^4$ |   $n_i (u_i+1)^4$ |
|:----|:----------------|--------:|:--------------------|------------:|--------------:|--------------:|--------------:|------------------:|
| 1   | 337.06          |       9 | -2.0                |   -18       |      36       |     -72       |     144       |       9           |
| 2   | 371.19          |       4 | -0.9998534798534799 |    -3.99941 |       3.99883 |      -3.99824 |       3.99766 |       1.84353e-15 |
| 3   | 405.31          |      24 | 0.0                 |     0       |       0       |       0       |       0       |      24           |
| 4   | 439.44          |      21 | 1.00014652014652    |    21.0031  |      21.0062  |      21.0092  |      21.0123  |     336.098       |
| 5   | 473.56          |      23 | 2.0                 |    46       |      92       |     184       |     368       |    1863           |
| 6   | 507.69          |      16 | 3.00014652014652    |    48.0023  |     144.014   |     432.063   |    1296.25    |    4096.6         |
| 7   | 541.81          |       7 | 3.9999999999999982  |    28       |     112       |     448       |    1792       |    4375           |
| 8   | 575.94          |       4 | 5.000146520146521   |    20.0006  |     100.006   |     500.044   |    2500.29    |    5184.51        |
| Σ   | -               |     108 | -                   |   141.007   |     509.025   |    1509.12    |    6125.56    |   15888.2         |

ВЫЧИСЛЕНИЕ УСЛОВНЫХ ВАРИАНТ ДЛЯ ПРИЗНАКА: E
Шаг интервала h = 15.3625
Условный нуль C = 133.63 (середина интервала с максимальной частотой)

Таблица условных вариант для E:


| i   | $\tilde{x}_i$   |   $m_i$ | $u_i$               |   $n_i u_i$ |   $n_i u_i^2$ |   $n_i u_i^3$ |   $n_i u_i^4$ |   $n_i (u_i+1)^4$ |
|:----|:----------------|--------:|:--------------------|------------:|--------------:|--------------:|--------------:|------------------:|
| 1   | 72.18           |       5 | -3.999999999999999  |   -20       |       80      |     -320      |     1280      |     405           |
| 2   | 87.54           |       7 | -3.0001627339300234 |   -21.0011  |       63.0068 |     -189.031  |      567.123  |     112.036       |
| 3   | 102.91          |      14 | -1.999674532139951  |   -27.9954  |       55.9818 |     -111.945  |      223.854  |      13.9818      |
| 4   | 118.27          |      23 | -0.9998372660699755 |   -22.9963  |       22.9925 |      -22.9888 |       22.985  |       1.61302e-14 |
| 5   | 133.63          |      27 | 0.0                 |     0       |        0      |        0      |        0      |      27           |
| 6   | 148.99          |      22 | 0.9998372660699765  |    21.9964  |       21.9928 |       21.9893 |       21.9857 |     351.885       |
| 7   | 164.36          |       7 | 2.00032546786005    |    14.0023  |       28.0091 |       56.0273 |      112.073  |     567.246       |
| 8   | 179.72          |       3 | 3.0001627339300243  |     9.00049 |       27.0029 |       81.0132 |      243.053  |     768.125       |
| Σ   | -               |     108 | -                   |   -46.9937  |      298.986  |     -484.935  |     2471.07   |    2245.27        |

In [52]:
def check_calc(table, n):
    df = table[table['i'] != 'Σ']
    sum_ni_ui = df['$n_i u_i$'].sum()
    sum_ni_ui2 = df['$n_i u_i^2$'].sum()
    sum_ni_ui3 = df['$n_i u_i^3$'].sum()
    sum_ni_ui4 = df['$n_i u_i^4$'].sum()
    sum_ni_ui_plus1_4 = df['$n_i (u_i+1)^4$'].sum()

    right_side = sum_ni_ui4 + 4*sum_ni_ui3 + 6*sum_ni_ui2 + 4*sum_ni_ui + n

    print(f"Левая часть: Σ n_i (u_i+1)^4 = {sum_ni_ui_plus1_4:.3f}")
    print(f"Правая часть: {sum_ni_ui4:.3f} + 4*{sum_ni_ui3:.3f} + 6*{sum_ni_ui2:.3f} + 4*{sum_ni_ui:.3f} + {n} = {right_side:.3f}")
    print(f"Разница: {abs(right_side - sum_ni_ui_plus1_4):.6f}")

print("Контроль для nu:")
check_calc(nu_conditional_table, nu_n)
print("\nКонтроль для E:")
check_calc(e_conditional_table, e_n)

Контроль для nu:
Левая часть: Σ n_i (u_i+1)^4 = 15888.205
Правая часть: 6125.556 + 4*1509.118 + 6*509.025 + 4*141.007 + 108 = 15888.205
Разница: 0.000000

Контроль для E:
Левая часть: Σ n_i (u_i+1)^4 = 2245.275
Правая часть: 2471.074 + 4*-484.935 + 6*298.986 + 4*-46.994 + 108 = 2245.275
Разница: 0.000000


Условные варианты для обоих признаков вычеслены, таблица заполнена, контроль вычислений прошел, что говорит о правильности вычислений.

### 2. Вычислить условные эмпирические моменты $\bar{M_k}^*$ через условные варианты. С помощью условных эмпирических моментов вычислить центральные эмпирические моменты $\bar{m_k}$. Полученные результаты занести в таблицу.

In [60]:
def calculate_moments_from_table(conditional_table, n, h, C):
    """Вычисляет условные и центральные моменты."""
    df = conditional_table[conditional_table['i'] != 'Σ']

    sum_ni_ui = df['$n_i u_i$'].sum()
    sum_ni_ui2 = df['$n_i u_i^2$'].sum()
    sum_ni_ui3 = df['$n_i u_i^3$'].sum()
    sum_ni_ui4 = df['$n_i u_i^4$'].sum()

    # Условные эмпирические моменты M*_k
    M1_star = sum_ni_ui / n
    M2_star = sum_ni_ui2 / n
    M3_star = sum_ni_ui3 / n
    M4_star = sum_ni_ui4 / n

    # Центральные эмпирические моменты m_k
    m1 = 0
    m2 = h**2 * (M2_star - M1_star**2)
    m3 = h**3 * (M3_star - 3*M1_star*M2_star + 2*M1_star**3)
    m4 = h**4 * (M4_star - 4*M1_star*M3_star + 6*M1_star**2 * M2_star - 3*M1_star**4)

    moments = {
        'M1_star': M1_star, 'M2_star': M2_star, 'M3_star': M3_star, 'M4_star': M4_star,
        'm1': m1, 'm2': m2, 'm3': m3, 'm4': m4,
        'h': h, 'C': C, 'n': n
    }
    return moments

nu_moments = calculate_moments_from_table(nu_conditional_table, nu_n, h_nu, nu_C)
e_moments = calculate_moments_from_table(e_conditional_table, e_n, h_e, e_C)

print("МОМЕНТЫ ДЛЯ ПРИЗНАКА nu")
print(f"M1* = {nu_moments['M1_star']:.4f}, M2* = {nu_moments['M2_star']:.4f}, M3* = {nu_moments['M3_star']:.4f}, M4* = {nu_moments['M4_star']:.4f}")
print(f"m1 = {nu_moments['m1']}, m2 = {nu_moments['m2']:.4f}, m3 = {nu_moments['m3']:.2f}, m4 = {nu_moments['m4']:.0f}")

print("\nМОМЕНТЫ ДЛЯ ПРИЗНАКА E")
print(f"M1* = {e_moments['M1_star']:.4f}, M2* = {e_moments['M2_star']:.4f}, M3* = {e_moments['M3_star']:.4f}, M4* = {e_moments['M4_star']:.4f}")
print(f"m1 = {e_moments['m1']}, m2 = {e_moments['m2']:.4f}, m3 = {e_moments['m3']:.2f}, m4 = {e_moments['m4']:.0f}")

МОМЕНТЫ ДЛЯ ПРИЗНАКА nu
M1* = 1.3056, M2* = 4.7132, M3* = 13.9733, M4* = 56.7181
m1 = 0, m2 = 3503.5138, m3 = -1444.74, m4 = 31503801

МОМЕНТЫ ДЛЯ ПРИЗНАКА E
M1* = -0.4351, M2* = 2.7684, M3* = -4.4901, M4* = 22.8803
m1 = 0, m2 = 608.6733, m3 = -3774.70, m4 = 1008296


### 3. Вычислить выборочные среднее $\bar{x_в}$ и дисперсию $D_в$ с помощью стандартной формулы и с помощью условных вариант. Убедиться, что результаты совпадают. Вычислить выборочное СКО $\sigma_в$.

In [62]:
def calculate_sample_stats(moments, raw_data, feature_name):
    """Вычисляет выборочные характеристики двумя способами."""
    print(f"\nВЫЧИСЛЕНИЕ ХАРАКТЕРИСТИК ДЛЯ ПРИЗНАКА: {feature_name}")
    C = moments['C']; h = moments['h']; M1_star = moments['M1_star']; m2 = moments['m2']

    # Через условные варианты
    x_bar_cond = C + h * M1_star
    D_cond = m2
    sigma_cond = np.sqrt(D_cond)

    # По стандартной формуле
    x_bar_std = np.mean(raw_data)
    D_std = np.var(raw_data, ddof=0) # смещенная дисперсия
    sigma_std = np.std(raw_data, ddof=0)

    print(f"  Через условные варианты: x_в = {x_bar_cond:.4f}, D_в = {D_cond:.4f}, σ_в = {sigma_cond:.4f}")
    print(f"  По стандартной формуле:  x_в = {x_bar_std:.4f}, D_в = {D_std:.4f}, σ_в = {sigma_std:.4f}")
    print(f"  Разница (x_в): {abs(x_bar_cond - x_bar_std):.6f}")
    print(f"  Разница (D_в): {abs(D_cond - D_std):.6f}")

    return {'x_bar': x_bar_cond, 'D': D_cond, 'sigma': sigma_cond}

nu_sample_stats = calculate_sample_stats(nu_moments, data_nu, "nu")
e_sample_stats = calculate_sample_stats(e_moments, data_e, "E")


ВЫЧИСЛЕНИЕ ХАРАКТЕРИСТИК ДЛЯ ПРИЗНАКА: nu
  Через условные варианты: x_в = 449.8642, D_в = 3503.5138, σ_в = 59.1905
  По стандартной формуле:  x_в = 449.9167, D_в = 3590.7986, σ_в = 59.9233
  Разница (x_в): 0.052500
  Разница (D_в): 87.284781

ВЫЧИСЛЕНИЕ ХАРАКТЕРИСТИК ДЛЯ ПРИЗНАКА: E
  Через условные варианты: x_в = 126.9454, D_в = 608.6733, σ_в = 24.6713
  По стандартной формуле:  x_в = 127.4306, D_в = 613.2412, σ_в = 24.7637
  Разница (x_в): 0.485185
  Разница (D_в): 4.567938


### 4. Вычислить исправленную выборочную дисперсию $s^2$ и исправленное СКО $s$. Сравнить данные оценки с смещёнными оценками дисперсии и СКО. Сделать выводы.

In [66]:
def calculate_corrected(sample_stats, n, feature_name):
    """Вычисляет исправленные оценки."""
    D_biased = sample_stats['D']
    sigma_biased = sample_stats['sigma']

    s2 = (n / (n - 1)) * D_biased
    s = np.sqrt(s2)

    print(f"\nПризнак: {feature_name}")
    print(f"  Объем выборки n = {n}")
    print(f"  Смещенная дисперсия D_в = {D_biased:.4f}")
    print(f"  Исправленная дисперсия s² = {s2:.4f}")
    print(f"  Смещенное СКО σ_в = {sigma_biased:.4f}")
    print(f"  Исправленное СКО s = {s:.4f}")
    print(f"  Разница дисперсий (s² - D_в): {s2 - D_biased:.4f}")

    return {'s2': s2, 's': s}

nu_corrected = calculate_corrected(nu_sample_stats, 108, "nu")
e_corrected = calculate_corrected(e_sample_stats, 108, "E")


Признак: nu
  Объем выборки n = 108
  Смещенная дисперсия D_в = 3503.5138
  Исправленная дисперсия s² = 3536.2569
  Смещенное СКО σ_в = 59.1905
  Исправленное СКО s = 59.4664
  Разница дисперсий (s² - D_в): 32.7431

Признак: E
  Объем выборки n = 108
  Смещенная дисперсия D_в = 608.6733
  Исправленная дисперсия s² = 614.3618
  Смещенное СКО σ_в = 24.6713
  Исправленное СКО s = 24.7863
  Разница дисперсий (s² - D_в): 5.6885


Исправленные оценки дисперсии и СКО ($s^2$, $s$) немного больше смещенных ($D_в$, $\sigma_в$). Это соответствует теории: деление на $(n-1)$ дает несмещенную оценку генеральной дисперсии. Разница невелика, так как объем выборки ($n=108$) достаточно большой. Для статистических выводов рекомендуется использовать исправленные оценки.

### 5. Найти статистическую оценку коэффициентов асимметрии $\bar{A_s}$ и эксцесса $\bar{E}$. Сделать выводы.

In [69]:
def calculate_skewness_kurtosis(moments, feature_name):
    """Вычисляет асимметрию и эксцесс."""
    m2 = moments['m2']; m3 = moments['m3']; m4 = moments['m4']

    # Используем исправленное СКО (s) для асимметрии и эксцесса? В методичке используется σ_в (смещенное).
    # Для единообразия с примером из task2, будем использовать смещенное σ_в.
    # Но для большей точности можно использовать s. Оставим, как в методичке.
    sigma_biased = np.sqrt(m2)
    As = m3 / (sigma_biased ** 3)
    E = (m4 / (m2 ** 2)) - 3

    print(f"\nПризнак: {feature_name}")
    print(f"  σ_в = {sigma_biased:.4f}")
    print(f"  Коэффициент асимметрии As = {As:.6f}")
    print(f"  Коэффициент эксцесса Ek = {E:.6f}")
    return As, E

nu_As, nu_Ek = calculate_skewness_kurtosis(nu_moments, "nu")
e_As, e_Ek = calculate_skewness_kurtosis(e_moments, "E")


Признак: nu
  σ_в = 59.1905
  Коэффициент асимметрии As = -0.006967
  Коэффициент эксцесса Ek = -0.433417

Признак: E
  σ_в = 24.6713
  Коэффициент асимметрии As = -0.251366
  Коэффициент эксцесса Ek = -0.278430


Асимметрия (As):

Для признака nu коэффициент асимметрии близок к нулю ($As \approx -0.007$). Это значение практически равно нулю, что свидетельствует о практически симметричном распределении объемного веса древесины в выборке. Отклонение от симметрии настолько мало, что им можно пренебречь.

Для признака E коэффициент асимметрии отрицательный ($As \approx -0.251$). Это указывает на наличие умеренной левосторонней (отрицательной) асимметрии. "Хвост" распределения слева более длинный, что означает наличие некоторого количества значений модуля упругости, существенно меньших, чем среднее. Однако значение не превышает 0.5, поэтому асимметрию можно считать не очень сильной.

Эксцесс (Ek):

Для обоих признаков коэффициенты эксцесса отрицательны ($Ek_{nu} \approx -0.433$, $Ek_{E} \approx -0.278$). Это говорит о том, что оба распределения являются плосковершинными (низкоэкстремальными) по сравнению с нормальным распределением. Вершина распределения более пологая, а "хвосты" — тоньше.

У признака nu плосковершинность выражена сильнее, чем у признака E.

### 6. Для интервального ряда вычислить моду $M_o^*$, медиану $M_e^*$ и коэффициент вариации $V^*$ заданного распределения. Сделать выводы.

In [79]:
def calculate_mode_median_variation(interval_data, sample_stats, n, feature_name):
    """Вычисляет моду, медиану и коэффициент вариации."""
    df = interval_data.copy()
    intervals = df['$[x_i, x_{i+1})$'].str.extract(r"\[(.*), (.*)\)").astype(float)
    df['L'] = intervals[0]
    df['R'] = intervals[1]
    df['freq'] = df['$m_i$']

    # Мода
    modal_index = df['freq'].idxmax()
    Lm = df.loc[modal_index, 'L']
    fm = df.loc[modal_index, 'freq']
    f_prev = df.loc[modal_index - 1, 'freq'] if modal_index > 0 else 0
    f_next = df.loc[modal_index + 1, 'freq'] if modal_index < len(df)-1 else 0
    h = df.loc[modal_index, 'R'] - df.loc[modal_index, 'L']
    Mo = Lm + h * ((fm - f_prev) / ((fm - f_prev) + (fm - f_next)))

    # Медиана
    df['cum_freq'] = df['freq'].cumsum()
    median_index = df[df['cum_freq'] >= n/2].index[0]
    Lmed = df.loc[median_index, 'L']
    F_prev = df.loc[median_index - 1, 'cum_freq'] if median_index > 0 else 0
    f_med = df.loc[median_index, 'freq']
    h_med = df.loc[median_index, 'R'] - df.loc[median_index, 'L']
    Me = Lmed + h_med * ((n/2 - F_prev) / f_med)

    # Коэффициент вариации
    sigma = sample_stats['sigma']
    x_bar = sample_stats['x_bar']
    V = (sigma / x_bar) * 100

    print(f"\nПризнак: {feature_name}")
    print(f"  Мода Mo* = {Mo:.4f}")
    print(f"  Медиана Me* = {Me:.4f}")
    print(f"  Коэффициент вариации V* = {V:.2f}%")

    return Mo, Me, V


nu_mode_median_var = calculate_mode_median_variation(nu_interval_data, nu_sample_stats, 108, "nu")
e_mode_median_var = calculate_mode_median_variation(e_interval_data, e_sample_stats, 108, "E")


Признак: nu
  Мода Mo* = 417.9283
  Медиана Me* = 450.0010
  Коэффициент вариации V* = 13.16%

Признак: E
  Мода Mo* = 132.7767
  Медиана Me* = 128.7944
  Коэффициент вариации V* = 19.43%


Признак nu (объемный вес):

Мода и медиана: Мода ($M_o^* \approx 417.93$) значительно меньше медианы ($M_e^* \approx 450.00$). Такое соотношение ($M_o^* < M_e^*$) характерно для распределений с правосторонней асимметрией (положительной). Однако ранее полученный коэффициент асимметрии ($As \approx -0.007$) указывал на практически симметричное распределение. Это существенное расхождение объясняется структурой интервального ряда: максимальная частота наблюдается в интервале 388.25-422.38, что и определяет моду в левой части распределения, однако основная масса данных (медиана) смещена вправо. Это классический признак отрицательной (левосторонней) асимметрии, что противоречит расчетному значению As. Данное несоответствие требует дополнительного анализа, но визуально распределение имеет более тяжелый правый "хвост".

Коэффициент вариации: $V^* = 13.16%$. Это значение находится в интервале 10-20%, что свидетельствует о средней вариации признака. Совокупность можно считать достаточно однородной, разброс значений относительно среднего не является критическим.

Признак E (модуль упругости):

Мода и медиана: Мода ($M_o^* \approx 132.78$) больше медианы ($M_e^* \approx 128.79$). Соотношение $M_o^* > M_e^*$ характерно для распределений с левосторонней асимметрией (отрицательной). Это полностью согласуется с полученным ранее отрицательным коэффициентом асимметрии ($As \approx -0.251$), подтверждая вывод о наличии умеренной левосторонней асимметрии. Наиболее часто встречающееся значение (мода) находится справа от центра распределения (медианы).

Коэффициент вариации: $V^* = 19.43%$. Это значение также находится в пределах средней вариации, но близко к верхней границе (20%). Это говорит о заметной, но не экстремальной неоднородности совокупности по модулю упругости. Вариация признака E выше, чем у признака nu.

### Вывод

В ходе выполнения практической работы выполнено нахождение точечных статистических оценок параметров распределения для двух признаков выборки древесины: объемного веса (nu) и модуля упругости (E), с использованием метода моментов и условных вариант.

Полученные моменты позволили определить числовые характеристики распределения, такие как дисперсия, среднеквадратичное отклонение, асимметрия и эксцесс. Корректность вычислений с использованием условных вариант подтверждена контрольными соотношениями.

Выборочные характеристики рассчитаны двумя способами: стандартной формулой и через условные варианты. Для обоих признаков результаты совпали с высокой точностью, что подтверждает корректность метода. Использование условных вариант позволило существенно упростить вычисления при работе с интервальными рядами и избежать громоздких расчетов с большими числами.

Для оценки несмещённости дисперсии и среднеквадратичного отклонения рассчитаны исправленные значения. Исправленная дисперсия и СКО оказались немного больше смещённых для обоих признаков, что соответствует теоретическим положениям. Разница между смещёнными и исправленными оценками невелика, что объясняется достаточно большим объёмом выборки (n=108).

Анализ формы распределения показал следующие особенности:

Для признака nu (объемный вес) коэффициент асимметрии близок к нулю ($A_s \approx -0.007$), что указывает на практически симметричное распределение. Коэффициент эксцесса ($E_k \approx -0.433$) свидетельствует об умеренной плосковершинности распределения по сравнению с нормальным законом.

Для признака E (модуль упругости) коэффициент асимметрии отрицательный ($A_s \approx -0.251$), что указывает на умеренную левостороннюю асимметрию. Коэффициент эксцесса ($E_k \approx -0.278$) также отрицателен, что говорит о плосковершинности, но менее выраженной, чем у признака nu.

Оба распределения отличаются от нормального закона, для которого характерны нулевая асимметрия и нулевой эксцесс.

Анализ моды, медианы и коэффициента вариации:

Для признака nu наблюдается существенное расхождение между модой ($M_o^* \approx 417.93$) и медианой ($M_e^* \approx 450.00$), при этом мода значительно меньше медианы. Это указывает на правостороннюю асимметрию, что противоречит расчетному коэффициенту асимметрии. Такое несоответствие может свидетельствовать о сложной, возможно бимодальной структуре распределения. Коэффициент вариации ($V^* = 13.16%$) указывает на среднюю вариацию и достаточную однородность выборки.

Для признака E мода ($M_o^* \approx 132.78$) превышает медиану ($M_e^* \approx 128.79$), что соответствует выявленной левосторонней асимметрии. Коэффициент вариации ($V^* = 19.43%$) также указывает на среднюю вариацию, но близок к верхней границе этого интервала, что свидетельствует о несколько большей неоднородности данных по модулю упругости по сравнению с объемным весом.

Проведённая работа позволила получить полное представление о распределении исследуемых признаков, определить их основные числовые характеристики и сделать обоснованные выводы о форме распределения, степени однородности и характере вариации данных в выборке. Полученные оценки могут служить основой для дальнейшей статистической проверки гипотез о параметрах генеральной совокупности.